# Les 07 - Planning Design Pattern

Deze notitieboek demonstreert het **Planning Design Pattern** voor AI-agenten met behulp van het Microsoft Agent Framework.
Je leert hoe je complexe reisaanvragen kunt opdelen in gestructureerde subtaken, deze kunt toewijzen aan specialistische agenten,
en het resulterende plan kunt uitvoeren — allemaal met gestructureerde output aangedreven door Pydantic-modellen.


## Installatie


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Taakontleding

Taakontleding is de kern van het planningsontwerppatroon. In plaats van één enkele agent te vragen een complexe aanvraag van begin tot eind af te handelen, breken we het probleem op in kleinere, duidelijk afgebakende **deeltaken**. Elke deeltaak wordt toegewezen aan een gespecialiseerde agent (bijv. vluchten, hotels, activiteiten) met duidelijke prioriteiten en afhankelijkheidsvolgorde.

Deze aanpak biedt verschillende voordelen:
- **Duidelijkheid**: elke deeltaak heeft een enkele verantwoordelijkheid.
- **Parallelisme**: onafhankelijke deeltaken kunnen gelijktijdig worden uitgevoerd.
- **Betrouwbaarheid**: fouten zijn geïsoleerd tot individuele deeltaken.
- **Budgetbewaking**: kosten worden per deeltaak geschat en opgeteld.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Een planningsagent maken met gestructureerde uitvoer

De planningsagent fungeert als een **receptionist coördinator**. Gegeven een high-level reisverzoek produceert hij een gestructureerd `TravelPlan` — waarbij het verzoek wordt opgedeeld in subtaken, prioriteiten worden gesteld, en afhankelijkheden worden geïdentificeerd zodat een conciërge of uitvoeringslaag het werk kan uitvoeren.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Een Plan Uitvoeren met Specialistische Hulpmiddelen

Zodra de reception agent een gestructureerd plan heeft opgesteld, voert de **concierge agent** het uit.
Elke specialistische tool behandelt één categorie van subtaken (vluchten, hotels, activiteiten). De concierge
doorloopt de subtaken van het plan in afhankelijkheidsvolgorde en geeft elke taak door aan het
geschikte hulpmiddel.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Samenvatting

In deze les heb je het **Planning Ontwerppatroon** voor AI-agenten geleerd:

1. **Taakontleding** — Een receptieplanagent splitst een complex reisverzoek op in gestructureerde subtaken met behulp van Pydantic-modellen, waarbij elke taak wordt toegewezen aan een specialistische agent met prioriteiten en afhankelijkheden.
2. **Gestructureerde Uitvoer** — Door een `response_format` mee te geven, retourneert de agent een gevalideerd `TravelPlan`-object in plaats van vrije tekst, waardoor verdere verwerking betrouwbaar wordt.
3. **Uitvoering van het Plan** — Een conciërge-agent doorloopt de subtaken met specialistische hulpmiddelen (`book_flight`, `reserve_hotel`, `book_activity`) om het plan uit te voeren en resultaten te rapporteren.

Dit patroon scheidt *wat er gedaan moet worden* (planning) van *hoe het gedaan wordt* (uitvoering), waardoor agenten modularer, beter testbaar en makkelijker uitbreidbaar zijn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI-vertalingsdienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel wij streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onjuistheden kunnen bevatten. Het originele document in de oorspronkelijke taal dient als gezaghebbende bron te worden beschouwd. Voor cruciale informatie wordt een professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
